In [100]:
from datasets import load_dataset
import pandas as pd
from langdetect import detect, LangDetectException
import numpy as np

ds1 = load_dataset("lavita/medical-qa-datasets", "all-processed")
ds2 = load_dataset("lavita/AlpaCare-MedInstruct-52k")
ds3 = load_dataset("lavita/MedREQAL")

df1 = ds1["train"].to_pandas()
df2 = ds2["train"].to_pandas()
df3 = ds3["train"].to_pandas()

print("Dataset1 columns:", df1.columns.tolist())
print("Dataset2 columns:", df2.columns.tolist())
print("Dataset3 columns:", df3.columns.tolist())
print("Counts:", len(df1), len(df2), len(df3))

columns = ["question", "background", "objective", "conclusion"]
df3 = df3[columns]
print("Dataset3 columns:", df3.columns.tolist())

Dataset1 columns: ['instruction', 'input', 'output', '__index_level_0__']
Dataset2 columns: ['output', 'input', 'instruction']
Dataset3 columns: ['question', 'background', 'objective', 'conclusion', 'verdicts', 'strength', 'label', 'category']
Counts: 239357 52002 2786
Dataset3 columns: ['question', 'background', 'objective', 'conclusion']


In [101]:
df3_mapped = pd.DataFrame()
df3_mapped["instruction"] = df3["question"].apply(lambda x: x.strip() if isinstance(x, str) else "")
df3_mapped["input"] = df3.apply(
    lambda row: "\n".join(filter(None, [
        row["background"].strip() if isinstance(row["background"], str) else "",
        row["objective"].strip() if isinstance(row["objective"], str) else ""
    ])), axis=1
)
df3_mapped["output"] = df3["conclusion"].apply(lambda x: x.strip() if isinstance(x, str) else "")


In [102]:
#
# REMOVE <noinput> AND QUOTATION MARKS + MERGE
#

df2 = df2.applymap(lambda x: x.strip('"').strip("'") if isinstance(x, str) else x)
df1 = df1.applymap(lambda x: x.strip('"').strip("'") if isinstance(x, str) else x)
df = pd.concat([df1, df2, df3_mapped], ignore_index=True)
df.drop('__index_level_0__', inplace=True, axis=1)


C:\Users\mati\AppData\Local\Temp\ipykernel_16444\2985220654.py:5: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df2 = df2.applymap(lambda x: x.strip('"').strip("'") if isinstance(x, str) else x)
C:\Users\mati\AppData\Local\Temp\ipykernel_16444\2985220654.py:6: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df1 = df1.applymap(lambda x: x.strip('"').strip("'") if isinstance(x, str) else x)


In [103]:
#
# REMOVE EMPTY ROWS
#

def removeEmpty(df):
    df['instruction'].replace('', np.nan, inplace=True)
    df['input'].replace('', np.nan, inplace=True)
    df['output'].replace('', np.nan, inplace=True)
    return df.dropna()

print(f"Before removal {len(df)}")
df_cleaned = removeEmpty(df)
print(f"After removal {len(df_cleaned)}")

pd.concat([df_cleaned,df]).drop_duplicates(keep=False)

Before removal 294145
After removal 293725


C:\Users\mati\AppData\Local\Temp\ipykernel_16444\2890651108.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['instruction'].replace('', np.nan, inplace=True)
C:\Users\mati\AppData\Local\Temp\ipykernel_16444\2890651108.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For ex

,instruction,input,output
4924,"If you are a doctor, please answer the medical...",NaN,"hello, 95% of the times' fever in this age gro..."
38944,"If you are a doctor, please answer the medical...",NaN,"hello, i have a seven day cycle. what day woul..."
74180,"If you are a doctor, please answer the medical...",NaN,"hi good day, welcome to chatbot. if your perio..."
77473,Answer this question truthfully,NaN,"I'm sorry, but the given answer ""Symmetric, we..."
80643,"If you are a doctor, please answer the medical...",NaN,hymen with an abnormal sperm count can conceiv...
134711,Answer this question truthfully,NaN,What is the name of the duct that drains the s...
135166,Answer this question truthfully,NaN,There is no vaccine to prevent Lassa fever. Pr...
155338,Answer this question truthfully,NaN,What is the name of the duct that drains the s...
163332,Answer this question truthfully,NaN,Lassa virus commonly involves the liver and re...
172949,"If you are a doctor, please answer the medical...",NaN,hellothanks for query. based n the facts that ...


In [104]:
# #
# # REMOVE NON ENGLISH ROWS
# #

# def is_english(text):
#     try:
#         return detect(text) == "en"
#     except LangDetectException:
#         return False

# mask_non_english = ~df_cleaned["instruction"].apply(is_english)
# removed_non_english = df_cleaned[mask_non_english]

# df_final = df_cleaned[~mask_non_english]

# print("Rows removed (non-English):", len(removed_non_english))
# print(len(df_final))
# removed_non_english.head(50)

In [ ]:
# #
# # REMOVE NON ENGLISH ROWS
# #

# from lingua import Language, LanguageDetectorBuilder

# detector = LanguageDetectorBuilder.from_languages(Language.ENGLISH).build()

# def is_english(text):
#     lang = detector.detect_language_of(text)
#     return lang == Language.ENGLISH

# mask_non_english = ~df_cleaned["instruction"].apply(is_english)
# removed_non_english = df_cleaned[mask_non_english]

# df_final = df_cleaned[~mask_non_english]

# print("Rows removed (non-English):", len(removed_non_english))
# print(len(df_final))

# from lingua import Language, LanguageDetectorBuilder

# detector = LanguageDetectorBuilder.from_languages(Language.ENGLISH).build()
# cols = ["instruction", "input", "output"]
# threshold = 0.6

# def row_is_english(row, min_length=50):
#     for c in cols:
#         val = row[c]
#         if isinstance(val, str) and len(val.strip()) >= min_length:
#             conf = detector.compute_language_confidence(val, Language.ENGLISH)
#             if conf < threshold:
#                 return False
#     return True

# mask_english = df_cleaned.apply(row_is_english, axis=1)
# df_final = df_cleaned[mask_english]
# removed_non_english = df_cleaned[~mask_english]

# print("Rows removed (non-English):", len(removed_non_english))
# print(len(df_final))
# removed_non_english.head(50)

In [106]:
#
# DROP DUPLICATES
#
df_final = df_cleaned
df_final = df_final.drop_duplicates(subset=["instruction", "output", "input"])
print(len(df_final))


293049


In [107]:
#
# Drop <noinput>
#

import re

def is_noinput_variant(x):
    if not isinstance(x, str):
        return False
    s = re.sub(r'[^a-zA-Z]', '', x).lower()
    return sorted(s) == sorted("noinput")

if "input" in df_final.columns:
    df_final["input"] = df_final["input"].replace("<noinput>", "").str.strip()
print(len(df_final))


def clean_val(x):
    if not isinstance(x, str):
        return ""
    x = x.strip().strip('"').strip("'")
    if is_noinput_variant(x):
        return ""
    return x

# Final rows not removed
df_final = df_final.applymap(clean_val)
# Final rows removed
df_fial_stripped = removeEmpty(df_final)

print(len(df_final))
print(len(df_fial_stripped))

293049

C:\Users\mati\AppData\Local\Temp\ipykernel_16444\3165563048.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final["input"] = df_final["input"].replace("<noinput>", "").str.strip()
C:\Users\mati\AppData\Local\Temp\ipykernel_16444\3165563048.py:27: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_final = df_final.applymap(clean_val)


C:\Users\mati\AppData\Local\Temp\ipykernel_16444\2890651108.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['instruction'].replace('', np.nan, inplace=True)
C:\Users\mati\AppData\Local\Temp\ipykernel_16444\2890651108.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For ex

293049
275991


In [114]:
df_final.head(10)
df_final.tail(10)


,instruction,input,output
294135,Can endoscopic scoring indices effectively eva...,Endoscopic assessment of mucosal disease activ...,"While the UCEIS, UCCIS and Mayo Clinic Endosco..."
294136,Is gefitinib effective and safe for the treatm...,The role of gefitinib for the treatment of adv...,"This systematic review shows that gefitinib, w..."
294137,Can quality improvement interventions effectiv...,Despite evidence supporting the effectiveness ...,The results of this review provide evidence th...
294138,Can anticonvulsant drugs effectively treat acu...,Tonic-clonic convulsions and convulsive status...,We have not identified any new high-quality ev...
294139,Can providing physicians with feedback about t...,Poor medication adherence decreases treatment ...,"Across nine studies, we observed little or no ..."
294140,Are prostanoids effective and safe in patients...,Peripheral arterial occlusive disease (PAOD) i...,We found high-quality evidence showing that pr...
294141,Does treating patients at home versus treating...,Deep vein thrombosis (DVT) occurs when a blood...,Low-quality evidence suggests that patients tr...
294142,Does radical multimodal treatment provide bene...,Malignant pleural mesothelioma is an almost al...,The overall strength of the evidence gathered ...
294143,Can propofol administration improve the quanti...,People in the intensive care unit (ICU) experi...,We found insufficient evidence to determine wh...
294144,Can anti-VEGF drugs be effective and safe for ...,Vascular endothelial growth factor (VEGF) play...,Implications for practice: Intravitreal bevaci...
